# 封裝 -- 保護你的資料

## 學習目標

- 理解為什麼需要封裝
- 分辨 Python 的三種存取層級：public、_protected、__private
- 理解 name mangling 機制
- 學會用 `@property` 控制屬性的讀寫

---

## 為什麼需要封裝？

沒有封裝的程式碼，任何人都能直接修改資料：

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

acc = BankAccount(1000)
acc.balance = -999999   # 餘額變成負的？沒人阻止你

封裝的目的：**防止物件進入不合理的狀態**。

---

## Python 的三種存取層級

Python 沒有像 Java 那樣的強制存取控制，而是靠**命名慣例**。

In [ ]:
class MyClass:
    def __init__(self):
        self.public = "誰都能存取"          # 公開
        self._protected = "請不要直接碰我"    # 保護（慣例）
        self.__private = "外面很難碰到我"     # 私有（name mangling）

### 實際測試

In [ ]:
obj = MyClass()
print(obj.public)        # OK
print(obj._protected)    # OK（Python 不會擋，但不建議）
# print(obj.__private)   # AttributeError!
print(obj._MyClass__private)  # OK（這就是 name mangling）

---

## Name Mangling 機制

雙底線開頭的屬性，Python 會自動改名：

```
__xxx  →  _ClassName__xxx
```

這不是真正的「私有」，而是**防止意外碰到**。你還是能透過改名後的名字存取，但這麼做表示你知道自己在幹嘛。

In [ ]:
class Secret:
    def __init__(self):
        self.__code = 1234

s = Secret()
# print(s.__code)             # AttributeError
print(s._Secret__code)        # 1234（刻意繞過）

---

## 用 @property 控制存取

`@property` 讓你把方法偽裝成屬性，在讀取和寫入時加上邏輯。

### 基本用法

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance     # 用 _ 表示不要直接碰

    @property
    def balance(self):
        """讀取餘額"""
        return self._balance

    @balance.setter
    def balance(self, value):
        """設定餘額（含驗證）"""
        if value < 0:
            raise ValueError("餘額不能為負數")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("存款金額必須為正")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("提款金額必須為正")
        if amount > self._balance:
            raise ValueError("餘額不足")
        self._balance -= amount

### 使用起來像普通屬性

In [ ]:
acc = BankAccount("小明", 1000)
print(acc.balance)     # 1000  ← 觸發 @property getter
acc.balance = 500      # OK    ← 觸發 @balance.setter
# acc.balance = -100   # ValueError: 餘額不能為負數

acc.deposit(200)
print(acc.balance)     # 700
# acc.withdraw(800)    # ValueError: 餘額不足

---

## 唯讀屬性

只定義 getter、不定義 setter，就是唯讀：

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.area)    # 78.53975
# c.area = 100   # AttributeError: can't set attribute

---

## 完整範例：溫度轉換

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius    # 觸發 setter 驗證

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("溫度不能低於絕對零度")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9   # 會觸發 celsius 的驗證

t = Temperature(100)
print(t.fahrenheit)    # 212.0
t.fahrenheit = 32
print(t.celsius)       # 0.0

---

## 存取層級比較表

| 命名 | 層級 | 外部可存取？ | 用途 |
|------|------|-------------|------|
| `name` | 公開 | 可以 | 一般屬性，對外開放 |
| `_name` | 保護（慣例） | 可以，但不建議 | 內部使用，提醒別人不要直接碰 |
| `__name` | 私有（mangling） | 需用 `_Class__name` | 防止子類別意外覆蓋 |
| `@property` | 受控存取 | 透過 getter/setter | 需要驗證或計算的屬性 |

---

## 本節重點

| 概念 | 說明 |
|------|------|
| 封裝的目的 | 防止物件進入不合理的狀態 |
| `_xxx` | 慣例上的保護屬性，Python 不強制 |
| `__xxx` | 觸發 name mangling，變成 `_Class__xxx` |
| `@property` | 把方法偽裝成屬性，加入讀寫控制 |
| 唯讀屬性 | 只定義 getter，不定義 setter |
| 驗證邏輯 | 放在 setter 中，確保資料永遠合法 |

---

> **下一篇：** [04_methods_and_properties.ipynb](./04_methods_and_properties.ipynb) -- 方法與屬性